In [ ]:
# Basic libraries
import os
import json
import re
import string
import pickle
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

# Text preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# CNN feature extraction
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model

# Example paths for MS COCO subset
CAPTIONS_FILE = '/content/annotations/captions_train2017.json'
IMAGES_DIR = '/content/train2017'

# Files to save preprocessing outputs
FEATURES_FILE = '/content/coco_image_features.pkl'
TOKENIZER_FILE = '/content/tokenizer.pkl'
MAPPINGS_FILE = '/content/vocab_mappings.pkl'

C:\Users\pc\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load MS COCO captions file
with open(CAPTIONS_FILE, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

# Build image_id -> file_name mapping
image_id_to_file = {img['id']: img['file_name'] for img in coco_data['images']}

# Collect captions grouped by image file name
captions_mapping = {}
for ann in coco_data['annotations']:
    image_id = ann['image_id']
    file_name = image_id_to_file[image_id]
    caption = ann['caption']
    captions_mapping.setdefault(file_name, []).append(caption)

print('Number of images with captions:', len(captions_mapping))
sample_file = next(iter(captions_mapping))
print('Sample image:', sample_file)
print('Sample captions:', captions_mapping[sample_file][:2])

Resuming download from 36700160 bytes (13643915080 bytes left)...
Resuming download to C:\Users\pc\.cache\kagglehub\datasets\hariwh0\ms-coco-dataset\1.archive (36700160/13680615240) bytes left.


  1%|          | 81.0M/12.7G [00:13<1:02:38, 3.62MB/s]


KeyboardInterrupt: 

In [ ]:
# Clean each caption in a simple standard way
def clean_caption(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep letters and spaces only
    text = re.sub(r'\s+', ' ', text).strip()
    words = [word for word in text.split() if len(word) > 1]
    return ' '.join(words)

# Add start/end tokens for sequence generation
cleaned_captions_mapping = {}
all_cleaned_captions = []

for file_name, captions in captions_mapping.items():
    cleaned_list = []
    for caption in captions:
        cleaned = clean_caption(caption)
        cleaned = '<start> ' + cleaned + ' <end>'
        cleaned_list.append(cleaned)
        all_cleaned_captions.append(cleaned)
    cleaned_captions_mapping[file_name] = cleaned_list

print('Total cleaned captions:', len(all_cleaned_captions))
print('Example cleaned caption:', all_cleaned_captions[0])


In [ ]:
# Tokenize processed captions
tokenizer = Tokenizer(oov_token='<unk>', filters='')
tokenizer.fit_on_texts(all_cleaned_captions)

# Convert captions into token sequences
caption_sequences = tokenizer.texts_to_sequences(all_cleaned_captions)

print('Vocabulary size from tokenizer:', len(tokenizer.word_index) + 1)
print('Sample tokenized caption:', caption_sequences[0])


In [ ]:
# === TASK 2: VOCABULARY CONSTRUCTION ===
# Build vocabulary with frequency filtering

# Count word frequencies
word_counts = Counter()
for caption in all_cleaned_captions:
    word_counts.update(caption.split())

# Apply frequency threshold (keep words appearing >= 5 times)
MIN_FREQUENCY = 5
frequent_words = {word: count for word, count in word_counts.items() if count >= MIN_FREQUENCY}

# Tokenize with filtered vocabulary
tokenizer = Tokenizer(num_words=len(frequent_words) + 1, oov_token='<unk>', filters='')
tokenizer.fit_on_texts(all_cleaned_captions)

# Build explicit mappings (CRITICAL: index 0 is padding)
word_to_index = {'<pad>': 0}  # Reserve index 0 for padding
word_to_index.update(tokenizer.word_index)  # Add tokenizer indices

index_to_word = {idx: word for word, idx in word_to_index.items()}
vocab_size = len(word_to_index)

# Verify special tokens
assert '<start>' in word_to_index, "ERROR: <start> not in vocabulary!"
assert '<end>' in word_to_index, "ERROR: <end> not in vocabulary!"

print('Vocabulary size:', vocab_size)
print('Index of <pad>:', word_to_index.get('<pad>'))
print('Index of <start>:', word_to_index.get('<start>'))
print('Index of <end>:', word_to_index.get('<end>'))

In [ ]:
# Save tokenizer and mappings for later use
with open(TOKENIZER_FILE, 'wb') as f:
    pickle.dump(tokenizer, f)

with open(MAPPINGS_FILE, 'wb') as f:
    pickle.dump({
        'word_to_index': word_to_index,
        'index_to_word': index_to_word,
        'vocab_size': vocab_size
    }, f)

print('Tokenizer and vocabulary mappings saved.')


In [ ]:
# Find maximum caption length for padding
max_caption_length = max(len(seq) for seq in caption_sequences)
print('Maximum caption length:', max_caption_length)


In [ ]:
# Prepare input-output sequences for caption generation preprocessing
# X_text: partial caption so far
# y_word: next word to predict
# X_image_names: matching image file name for each text sequence

X_text = []
y_word = []
X_image_names = []

for file_name, captions in cleaned_captions_mapping.items():
    for caption in captions:
        seq = tokenizer.texts_to_sequences([caption])[0]

        # Create many training pairs from one caption
        for i in range(1, len(seq)):
            in_seq = seq[:i]
            out_seq = seq[i]
            X_text.append(in_seq)
            y_word.append(out_seq)
            X_image_names.append(file_name)

# Pad input text sequences to same length
X_text = pad_sequences(X_text, maxlen=max_caption_length, padding='post')
y_word = np.array(y_word)
X_image_names = np.array(X_image_names)

print('Text input shape:', X_text.shape)
print('Target output shape:', y_word.shape)
print('Image reference shape:', X_image_names.shape)


In [ ]:
# Show one prepared example
example_index = 0
print('Image file:', X_image_names[example_index])
print('Padded input sequence:', X_text[example_index])
print('Target next word index:', y_word[example_index])
print('Target next word:', index_to_word.get(int(y_word[example_index]), '<unknown>'))


In [ ]:
# === TASK 4: IMAGE FEATURE EXTRACTION (CNN) ===
# Extract features using ResNet50 (consistent with PyTorch model)

import torch
import torchvision.models as models
from torchvision import transforms

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load pretrained ResNet50
resnet50 = models.resnet50(pretrained=True)

# Remove classification layer - keep feature extractor
modules = list(resnet50.children())[:-1]
feature_extractor = torch.nn.Sequential(*modules)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# Freeze parameters (transfer learning - no fine-tuning)
for param in feature_extractor.parameters():
    param.requires_grad = False

print('CNN model ready (ResNet50)')
print('Output feature dimension: 2048')

# Preprocessing pipeline for ImageNet
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

In [ ]:
# Function to extract one image feature vector
def extract_image_feature(img_path, model, preprocessing, device):
    from PIL import Image
    try:
        img = Image.open(img_path).convert('RGB')
        img_tensor = preprocessing(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            features = model(img_tensor)
        
        return features.flatten().cpu().numpy()
    except:
        return None

In [ ]:
# Extract and store one feature vector per image
image_features = {}
failed_count = 0

for file_name in tqdm(cleaned_captions_mapping.keys()):
    img_path = os.path.join(IMAGES_DIR, file_name)
    if os.path.exists(img_path):
        feature = extract_image_feature(img_path, feature_extractor, preprocess, device)
        if feature is not None:
            image_features[file_name] = feature
        else:
            failed_count += 1
    else:
        failed_count += 1

print('Extracted features for images:', len(image_features))
print('Failed/missing images:', failed_count)

# Remove images without features from training set
for img in list(cleaned_captions_mapping.keys()):
    if img not in image_features:
        del cleaned_captions_mapping[img]

In [ ]:
# Save extracted feature vectors
with open(FEATURES_FILE, 'wb') as f:
    pickle.dump(image_features, f)

print('Image features saved to:', FEATURES_FILE)


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        
        # Load a pre-trained ResNet50 model to extract visual features
        # We use pre-trained weights to leverage features learned from ImageNet
        resnet = models.resnet50(pretrained=True)
        
        # Remove the final fully connected classification layer
        # We only want the feature extraction layers, not the ImageNet classes
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        
        # Add a linear layer to map the extracted visual features to the desired embedding size
        # This ensures the image features match the dimension of the word embeddings later
        self.embed = nn.Linear(resnet.fc.in_features, embed_size)
        
        # Optional: Batch Normalization helps stabilize and speed up training
        self.batch_norm = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        # Pass the raw images through the modified ResNet
        features = self.resnet(images)
        
        # Flatten the features from a multi-dimensional tensor to a 2D tensor (batch_size, features)
        features = features.view(features.size(0), -1)
        
        # Project the features into the embedding space
        features = self.embed(features)
        features = self.batch_norm(features)
        
        return features

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1, dropout=0.5):
        super(DecoderRNN, self).__init__()
        
        # Step (b): Embedding Layer
        # Convert word indices to dense vectors; padding_idx=0 ensures padding tokens produce zero vectors
        self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        nn.init.uniform_(self.embed.weight, -0.05, 0.05)
        self.embed_norm = nn.LayerNorm(embed_size)
        
        # Step (c): RNN Decoder (LSTM)
        # LSTM with dropout for regularization
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True, 
                           dropout=dropout if num_layers > 1 else 0.0)
        
        # Step (d): Final Dense Layer
        # Project LSTM output to vocabulary probabilities
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        # Remove <end> token from captions
        captions = captions[:, :-1]
        
        # Convert token indices to embedding vectors
        embeddings = self.embed(captions)
        embeddings = self.embed_norm(embeddings)
        
        # Reshape image features to match sequence dimensions
        features = features.unsqueeze(1)
        
        # Concatenate image features with word embeddings
        inputs = torch.cat((features, embeddings), dim=1)
        
        # Pass through LSTM
        lstm_out, _ = self.lstm(inputs)
        
        # Project to vocabulary
        outputs = self.linear(lstm_out)
        
        return outputs

In [ ]:
# Hyperparameters (Example values)
embed_size = 256
hidden_size = 512
vocab_size = 10000 # This should match the length of the vocabulary you built in step (b)
num_layers = 1

# Initialize the Encoder (CNN) and Decoder (RNN)
encoder = EncoderCNN(embed_size)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

# Example Forward Pass logic (used during your training loop)
# images: (batch_size, 3, 224, 224) 
# captions: (batch_size, max_seq_length)

# 1. Extract visual features
# features = encoder(images)

# 2. Generate caption predictions
# outputs = decoder(features, captions)